# AI-Powered Retail Analytics & Intelligent Customer Churn Predictor

**Project Type:** Machine Learning + Retail Analytics  
**Dataset:** UCI Online Retail  
**Goal:** Analyze retail transactions, create customer-level RFM/business features, segment customers, and predict whether a customer is likely to churn.

### Main modules
1. Data acquisition and cleaning
2. Retail sales analytics
3. Customer RFM analysis
4. Customer segmentation
5. Churn-label creation using a time-based holdout period
6. Machine-learning churn prediction
7. Model evaluation
8. Feature importance and business recommendations
9. Single-customer churn prediction utility

> **Important:** The churn model avoids target leakage by building customer features only from the observation period and defining churn from behavior in the following 90-day period.


## 1. Install / Import Libraries

If you are running this notebook locally, install the packages listed in `requirements.txt`.

**Python:** 3.10+ recommended.


In [ ]:
# If needed, uncomment the next line in Jupyter/Colab:
# %pip install -r requirements.txt

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)


## 2. Load the UCI Online Retail Dataset

The project uses the public **UCI Online Retail** dataset. It contains transactions from a UK-based online retailer from 01-Dec-2010 to 09-Dec-2011.

The notebook first tries `ucimlrepo`. If internet access/package retrieval is unavailable, it can fall back to a local `Online Retail.xlsx` file placed beside the notebook.


In [ ]:
DATA_FILE = "Online Retail.xlsx"

try:
    from ucimlrepo import fetch_ucirepo
    online_retail = fetch_ucirepo(id=352)
    raw = online_retail.data.features.copy()
    print("Loaded dataset from UCI using ucimlrepo.")
except Exception as e:
    print("UCI API loading was unavailable:", e)
    print("Trying local Excel file:", DATA_FILE)
    raw = pd.read_excel(DATA_FILE)

print("Raw shape:", raw.shape)
display(raw.head())
print("\nColumns:", list(raw.columns))


## 3. Standardize Column Names and Clean the Transactions

The original UCI fields are typically:
`InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country`.

Rows with missing `CustomerID` are excluded from customer-level churn modeling. Cancelled invoices and non-positive quantities/prices are excluded from positive-sales analytics.


In [ ]:
# Normalize column names across possible UCI loaders
rename_map = {}
for c in raw.columns:
    key = str(c).strip().lower().replace(" ", "").replace("_", "")
    if key == "invoiceno":
        rename_map[c] = "InvoiceNo"
    elif key == "stockcode":
        rename_map[c] = "StockCode"
    elif key == "description":
        rename_map[c] = "Description"
    elif key == "quantity":
        rename_map[c] = "Quantity"
    elif key in ("invoicedate", "invoicedatetime"):
        rename_map[c] = "InvoiceDate"
    elif key in ("unitprice", "price"):
        rename_map[c] = "UnitPrice"
    elif key in ("customerid", "customer_id"):
        rename_map[c] = "CustomerID"
    elif key == "country":
        rename_map[c] = "Country"

raw = raw.rename(columns=rename_map)

required = ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw.copy()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

df = df.dropna(subset=["InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice", "CustomerID"])
df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["IsCancellation"] = df["InvoiceNo"].str.upper().str.startswith("C")
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

sales = df[
    (~df["IsCancellation"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0) &
    (df["Revenue"] > 0)
].copy()

print("Clean sales shape:", sales.shape)
print("Date range:", sales["InvoiceDate"].min(), "to", sales["InvoiceDate"].max())
print("Unique customers:", sales["CustomerID"].nunique())
print("Unique invoices:", sales["InvoiceNo"].nunique())


## 4. Retail Analytics Dashboard Metrics

We calculate overall revenue, orders, customers, products, monthly revenue, top countries, and top products.


In [ ]:
total_revenue = sales["Revenue"].sum()
total_orders = sales["InvoiceNo"].nunique()
total_customers = sales["CustomerID"].nunique()
total_products = sales["StockCode"].nunique()

kpi = pd.DataFrame({
    "Metric": ["Total Revenue", "Orders", "Customers", "Products"],
    "Value": [total_revenue, total_orders, total_customers, total_products]
})
display(kpi)

monthly = (
    sales.assign(Month=sales["InvoiceDate"].dt.to_period("M").dt.to_timestamp())
    .groupby("Month", as_index=False)["Revenue"].sum()
)

plt.figure(figsize=(12, 5))
plt.plot(monthly["Month"], monthly["Revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

top_countries = (
    sales.groupby("Country")["Revenue"].sum()
    .sort_values(ascending=False).head(10)
)
plt.figure(figsize=(10, 5))
top_countries.sort_values().plot(kind="barh")
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue")
plt.tight_layout()
plt.show()

top_products = (
    sales.groupby("StockCode")["Revenue"].sum()
    .sort_values(ascending=False).head(10)
)
display(top_products.to_frame("Revenue"))


## 5. Build a Time-Based Churn Dataset

### Observation period
All purchases up to **90 days before the final transaction date** are used to build customer features.

### Prediction period
The next 90 days are used only to determine the target:
- `Churn = 1`: customer made **no purchase** in the future 90-day period.
- `Churn = 0`: customer made at least one purchase in that period.

This is a more realistic supervised-learning setup than defining churn directly from the same features used for prediction.


In [ ]:
max_date = sales["InvoiceDate"].max().normalize()
cutoff_date = max_date - pd.Timedelta(days=90)
future_end = max_date

observation = sales[sales["InvoiceDate"] <= cutoff_date].copy()
future = sales[(sales["InvoiceDate"] > cutoff_date) & (sales["InvoiceDate"] <= future_end)].copy()

print("Observation end:", cutoff_date)
print("Future period:", cutoff_date, "to", future_end)
print("Observation rows:", len(observation))
print("Future rows:", len(future))


## 6. Customer-Level Feature Engineering

Features include:
- **Recency:** days since last purchase at the observation cutoff
- **Frequency:** number of unique invoices
- **Monetary:** total revenue
- Average order value
- Total items purchased
- Unique products
- Average quantity per line
- Active purchase days
- Cancellation count, where available
- Country

These features are useful for both RFM analysis and churn modeling.


In [ ]:
reference_date = cutoff_date + pd.Timedelta(days=1)

obs_group = observation.groupby("CustomerID")

customer_features = obs_group.agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
    TotalItems=("Quantity", "sum"),
    UniqueProducts=("StockCode", "nunique"),
    AvgQuantity=("Quantity", "mean"),
    ActiveDays=("InvoiceDate", lambda x: x.dt.date.nunique()),
    Country=("Country", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
).reset_index()

customer_features["Recency"] = (
    reference_date - customer_features["LastPurchase"]
).dt.days

customer_features["AvgOrderValue"] = (
    customer_features["Monetary"] / customer_features["Frequency"]
)

# Count cancellations linked to each customer in the original cleaned dataset.
cancel = df[
    df["IsCancellation"] &
    (df["InvoiceDate"] <= cutoff_date)
].groupby("CustomerID").size().rename("CancellationCount")

customer_features = customer_features.merge(cancel, on="CustomerID", how="left")
customer_features["CancellationCount"] = customer_features["CancellationCount"].fillna(0)

# Churn target from the future period
future_customers = set(future["CustomerID"].unique())
customer_features["Churn"] = (
    ~customer_features["CustomerID"].isin(future_customers)
).astype(int)

customer_features = customer_features.replace([np.inf, -np.inf], np.nan).dropna()

print("Customer feature table:", customer_features.shape)
display(customer_features.head())
print("\nChurn distribution:")
display(customer_features["Churn"].value_counts(normalize=True).rename("Proportion"))


## 7. RFM Segmentation

RFM means:
- **R — Recency:** how recently the customer purchased
- **F — Frequency:** how often the customer purchased
- **M — Monetary:** how much revenue the customer generated

Lower recency is better, while higher frequency and monetary value are better. Quintile scores are combined into simple business segments.


In [ ]:
rfm = customer_features[[
    "CustomerID", "Recency", "Frequency", "Monetary", "Churn"
]].copy()

rfm["R_Score"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5,4,3,2,1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

def rfm_segment(score):
    if score >= 13:
        return "Champions"
    if score >= 10:
        return "Loyal / High Value"
    if score >= 7:
        return "Potential / Regular"
    if score >= 5:
        return "At Risk"
    return "Hibernating"

rfm["Segment"] = rfm["RFM_Score"].apply(rfm_segment)

segment_summary = (
    rfm.groupby("Segment")
    .agg(Customers=("CustomerID", "nunique"),
         Revenue=("Monetary", "sum"),
         AvgRecency=("Recency", "mean"),
         AvgFrequency=("Frequency", "mean"))
    .sort_values("Revenue", ascending=False)
)

display(segment_summary)

plt.figure(figsize=(10, 5))
segment_summary["Customers"].sort_values().plot(kind="barh")
plt.title("Customer Segments")
plt.xlabel("Number of Customers")
plt.tight_layout()
plt.show()


## 8. Prepare Data for Machine Learning

Categorical feature: `Country`  
Numeric features: RFM and behavioral features.

A `ColumnTransformer` and `Pipeline` keep preprocessing and model training together.


In [ ]:
feature_cols = [
    "Recency", "Frequency", "Monetary", "TotalItems",
    "UniqueProducts", "AvgQuantity", "ActiveDays",
    "AvgOrderValue", "CancellationCount", "Country"
]

X = customer_features[feature_cols].copy()
y = customer_features["Churn"].copy()

numeric_features = [
    "Recency", "Frequency", "Monetary", "TotalItems",
    "UniqueProducts", "AvgQuantity", "ActiveDays",
    "AvgOrderValue", "CancellationCount"
]
categorical_features = ["Country"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

model = RandomForestClassifier(
    n_estimators=350,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
proba = pipeline.predict_proba(X_test)[:, 1]


## 9. Evaluate the Churn Model


In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred, zero_division=0),
    "F1 Score": f1_score(y_test, pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, proba)
}
metrics_df = pd.DataFrame([metrics]).T.rename(columns={0: "Score"})
display(metrics_df)

print("Classification Report")
print(classification_report(y_test, pred, target_names=["Not Churned", "Churned"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred,
    display_labels=["Not Churned", "Churned"],
    cmap=None
)
plt.title("Churn Confusion Matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, proba)
plt.title("ROC Curve")
plt.tight_layout()
plt.show()


## 10. Feature Importance

The Random Forest model helps identify which customer attributes contribute most to churn predictions. These are predictive associations, not proof of causation.


In [ ]:
# Retrieve transformed feature names
pre = pipeline.named_steps["preprocessor"]
rf = pipeline.named_steps["model"]

feature_names = pre.get_feature_names_out()
importance = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

display(importance.head(15).to_frame("Importance"))

plt.figure(figsize=(10, 6))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Model Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 11. Customer Risk Scoring

The following table converts model probabilities into simple operational risk bands.

- **High Risk:** probability >= 0.70
- **Medium Risk:** 0.40–0.69
- **Low Risk:** < 0.40

These thresholds are business-rule examples and should be tuned using validation data and campaign capacity.


In [ ]:
risk_output = customer_features[["CustomerID", "Country", "Recency", "Frequency", "Monetary", "Churn"]].copy()
risk_output["ChurnProbability"] = pipeline.predict_proba(
    customer_features[feature_cols]
)[:, 1]

risk_output["RiskBand"] = pd.cut(
    risk_output["ChurnProbability"],
    bins=[-0.01, 0.40, 0.70, 1.00],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

display(
    risk_output.sort_values("ChurnProbability", ascending=False).head(20)
)

print("\nRisk-band counts:")
display(risk_output["RiskBand"].value_counts())


## 12. Single-Customer Prediction Function

Use this function after training to estimate churn probability for a new customer's feature record.


In [ ]:
def predict_customer_churn(customer_record: dict):
    """Return churn probability and risk band for one customer."""
    row = pd.DataFrame([customer_record])
    probability = float(pipeline.predict_proba(row)[0, 1])
    if probability >= 0.70:
        risk = "High Risk"
    elif probability >= 0.40:
        risk = "Medium Risk"
    else:
        risk = "Low Risk"
    return {
        "churn_probability": round(probability, 4),
        "risk_band": risk
    }

# Example
example_customer = {
    "Recency": 45,
    "Frequency": 5,
    "Monetary": 250.0,
    "TotalItems": 80,
    "UniqueProducts": 15,
    "AvgQuantity": 2.5,
    "ActiveDays": 5,
    "AvgOrderValue": 50.0,
    "CancellationCount": 0,
    "Country": "United Kingdom"
}

print(predict_customer_churn(example_customer))


## 13. Business Recommendations

The model can support—not replace—business decisions.

Examples:
- High-risk, high-value customers can be considered for retention campaigns.
- Low-recency and low-frequency customers can receive re-engagement messages.
- Champions can receive loyalty benefits or personalized product recommendations.
- Product and country trends can guide inventory and marketing analysis.
- Model probabilities should be monitored over time because customer behavior and business conditions change.

### Limitations
- Churn is operationally defined as no purchase during the selected 90-day future period.
- The dataset represents one retailer and one historical time window.
- Predictions are associations and should not be interpreted as causal explanations.
- The model does not include marketing exposure, customer demographics, complaints, or external economic variables.
- Thresholds should be tuned to the retailer's campaign cost and capacity.


## 14. Save Outputs

This optional section saves the customer risk table and model metrics for use in a report or dashboard.


In [ ]:
risk_output.to_csv("customer_churn_risk_output.csv", index=False)
metrics_df.to_csv("model_metrics.csv")

print("Saved:")
print("- customer_churn_risk_output.csv")
print("- model_metrics.csv")


## 15. Project Completion Checklist

- [x] Retail transaction cleaning
- [x] Sales and revenue analytics
- [x] Customer RFM features
- [x] Customer segmentation
- [x] Time-based churn target
- [x] Random Forest churn model
- [x] Evaluation metrics
- [x] Feature importance
- [x] Customer risk scoring
- [x] Single-customer prediction function
- [x] CSV output generation

**Dataset source:** UCI Machine Learning Repository — Online Retail, Chen (2015), DOI 10.24432/C5BW33.
